# Time Series Fundamentals

Companion notebook for the [Time Series Fundamentals lesson](https://ml-viz-ruby.vercel.app/courses/time-series/01-time-series-fundamentals).

**The idea in one sentence.** A time series is data *ordered in time*, so its
observations are **correlated with their own past** — which breaks the i.i.d.
assumption behind ordinary ML and forces its own toolkit: decomposition,
**stationarity**, and the **autocorrelation function (ACF)**.

Three foundational pieces, from scratch:

- **Decomposition** into trend + seasonality + residual — the mental model for
  every classical method.
- **Stationarity** — most models assume the statistical properties don't drift;
  differencing/log-transforms make a series stationary.
- **The ACF** — how correlated each observation is with the one $k$ steps back,
  the fingerprint used to identify models.

We build the ACF from scratch, **validate it and the stationarity transform**,
then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
np.random.seed(42)

## Generating synthetic series with 4 components

In [ ]:
t = np.arange(48)
trend = 100 + 2.1 * t
seasonal = 25 * np.sin(2 * np.pi * t / 12)
residual = np.random.normal(0, 7, 48)
observed = trend + seasonal + residual

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
for ax, y, title in zip(axes, [observed, trend, seasonal, residual],
                         ['Observed', 'Trend', 'Seasonal', 'Residual']):
    ax.plot(t, y, color='#2dd4bf', linewidth=1.5)
    ax.set_title(title, color='white', fontsize=11)
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Stationarity: ADF test

In [ ]:
try:
    from statsmodels.tsa.stattools import adfuller
    raw_result = adfuller(observed)
    print(f"Raw series  — ADF stat: {raw_result[0]:.3f}, p-value: {raw_result[1]:.4f}")
    
    log_diff = np.diff(np.log(observed + 1e-9))
    diff_result = adfuller(log_diff)
    print(f"Log+diff    — ADF stat: {diff_result[0]:.3f}, p-value: {diff_result[1]:.4f}")
    print("Stationary after transformation!" if diff_result[1] < 0.05 else "Still non-stationary")
except ImportError:
    print("statsmodels not installed — run: pip install statsmodels")
    # Manual check: compute rolling mean/variance
    half = len(observed) // 2
    print(f"Mean first half: {observed[:half].mean():.1f}, second half: {observed[half:].mean():.1f}")
    print(f"Std first half:  {observed[:half].std():.1f},  second half: {observed[half:].std():.1f}")

### Validate: differencing makes the series stationary

A stationary series has a roughly *constant mean over time*. The raw series has a
strong trend, so its first-half and second-half means differ wildly; after
log-differencing, the two halves should have nearly the same mean. That mean
stability is the numpy-only stationarity check (the ADF test formalises it).

In [ ]:
def mean_shift(y):
    h = len(y) // 2
    return abs(y[:h].mean() - y[h:].mean())

raw_shift = mean_shift(observed)
log_diff = np.diff(np.log(observed + 1e-9))
diff_shift = mean_shift(log_diff)
print(f'raw series   half-to-half mean shift: {raw_shift:.2f}  (large -> non-stationary)')
print(f'log-differenced mean shift:           {diff_shift:.4f}  (~0 -> stationary)')
assert raw_shift > 1.0 and diff_shift < 0.1, 'differencing should flatten the mean'
print('\n✅ differencing removes the trend and stabilises the mean (stationary)')

## Computing sample ACF from scratch

In [ ]:
def sample_acf(y, max_lag=16):
    y_centered = y - y.mean()
    c0 = np.dot(y_centered, y_centered)
    return np.array([np.dot(y_centered[:len(y)-k], y_centered[k:]) / c0
                     for k in range(1, max_lag + 1)])

acf_vals = sample_acf(observed)
lags = np.arange(1, 17)
sig_band = 1.96 / np.sqrt(len(observed))

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(lags, acf_vals, color='#818cf8', alpha=0.8)
ax.axhline(sig_band, color='#f59e0b', linestyle='--', linewidth=1, label='±95% band')
ax.axhline(-sig_band, color='#f59e0b', linestyle='--', linewidth=1)
ax.set_xlabel('Lag', color='white')
ax.set_ylabel('ACF', color='white')
ax.set_title('Sample Autocorrelation Function', color='white')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### Validate: our sample ACF matches the direct definition

The sample ACF at lag $k$ is the normalised lagged covariance
$\rho_k = \frac{\sum_t (y_t-\bar y)(y_{t-k}-\bar y)}{\sum_t (y_t-\bar y)^2}$.
We cross-check our vectorised implementation against an explicit loop, and confirm
$\rho_0 = 1$ by construction.

In [ ]:
def acf_reference(y, max_lag):
    yc = y - y.mean()
    c0 = np.sum(yc ** 2)
    out = []
    for k in range(1, max_lag + 1):
        s = sum(yc[t] * yc[t - k] for t in range(k, len(y)))
        out.append(s / c0)
    return np.array(out)

ref = acf_reference(observed, 16)
print(f'max |vectorised - explicit| ACF: {np.abs(acf_vals - ref).max():.2e}')
assert np.allclose(acf_vals, ref), 'the vectorised ACF must match the explicit definition'
# lag-0 autocorrelation is exactly 1
assert np.isclose(np.dot(observed - observed.mean(), observed - observed.mean()) /
                  np.dot(observed - observed.mean(), observed - observed.mean()), 1.0)
print('✅ sample ACF matches its definition (and rho_0 = 1)')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **non-stationarity** | most models assume stable mean/variance; trends and unit roots break them |
| **spurious regression** | two independent trending series look strongly correlated (demo below) |
| **over-differencing** | differencing a stationary series injects negative autocorrelation |
| **seasonality** | a strong seasonal cycle shows up as ACF spikes at the period; may need seasonal differencing |
| **ACF significance band** | $\pm 1.96/\sqrt{n}$ is only a rough guide; multiple lags inflate false positives |

Demo: spurious correlation between two independent random walks.

In [ ]:
# Spurious regression: two INDEPENDENT random walks can show a huge correlation purely
# because both trend — the classic trap that makes non-stationarity dangerous for modelling.
rng = np.random.default_rng(0)
def random_walk(n): return np.cumsum(rng.normal(size=n))
corrs = [abs(np.corrcoef(random_walk(200), random_walk(200))[0, 1]) for _ in range(200)]
print(f'mean |correlation| between INDEPENDENT random walks: {np.mean(corrs):.2f}')
print(f'fraction with |correlation| > 0.5: {np.mean(np.array(corrs) > 0.5):.0%}')
print('\nTwo unrelated trending series look strongly correlated -> always difference to')
print('stationarity BEFORE regressing, or you will "discover" relationships that do not exist.')

## ✏️ Your turn: Generate an AR(1) series

In [ ]:
# TODO(you): Generate an AR(1) series with phi=0.5 and n=100
# y[t] = 0.5 * y[t-1] + noise, noise ~ N(0, 1)

# YOUR CODE HERE
y_ar1 = None  # replace

assert y_ar1 is not None, "Assign y_ar1"
assert len(y_ar1) == 100, "Should have 100 observations"
print("✓ AR(1) series generated")

<details><summary>Solution</summary>

```python
rng = np.random.default_rng(42)
n = 100
y_ar1 = np.zeros(n)
for t in range(1, n):
    y_ar1[t] = 0.5 * y_ar1[t-1] + rng.normal(0, 1)
```
</details>

## Key takeaways

- **Time series are correlated with their past**, so i.i.d. ML assumptions break —
  you need decomposition, stationarity, and the ACF.
- **Decomposition** = trend + seasonality + residual, the model behind classical
  forecasting.
- **Stationarity matters:** differencing/log-transforms stabilise the mean (we
  verified the half-to-half mean shift collapses).
- **The ACF is the fingerprint** — we matched its definition exactly; it's how the
  next lesson identifies AR vs MA models.
- **Beware spurious regression:** two independent trending series look correlated —
  difference first.